# 1D CNN Exercise with Financial Data

This notebook builds a **1D Convolutional Neural Network (Conv1D)** in **TensorFlow 2** to predict whether the **next-day return** of a financial asset will be **positive or negative**.

## Learning goals

- understand how **Conv1D** works on time series
- transform a financial series into **supervised learning windows**
- train and evaluate a **binary classifier**
- discuss why financial prediction is difficult


## Prediction task

At each time step $t$, use the previous **20 daily returns** to predict:

$$
y_t = \begin{cases}
1 & \text{if } r_{t+1} > 0 \\
0 & \text{if } r_{t+1} \le 0
\end{cases}
$$

We will use **SPY** (S&P 500 ETF) from Yahoo Finance.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)

## 2. Download financial data

This notebook uses `yfinance`. Install it first if needed:

```python
!pip install yfinance
```

In [ ]:
import yfinance as yf

ticker = 'SPY'
df = yf.download(ticker, start='2015-01-01', end='2025-01-01')
df.head()

## 3. Compute daily returns

In [ ]:
df['return'] = df['Close'].pct_change()
df = df.dropna()

df[['Close', 'return']].head()

## 4. Visualize the time series

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df.index, df['return'])
plt.title('Daily Returns of SPY')
plt.xlabel('Date')
plt.ylabel('Return')
plt.show()

## 5. Create rolling windows

We will use the last **20 returns** as input features.

In [ ]:
window_size = 20

X = []
y = []

returns = df['return'].values

for i in range(window_size, len(returns) - 1):
    X.append(returns[i-window_size:i])
    y.append(1 if returns[i+1] > 0 else 0)

X = np.array(X)
y = np.array(y)

print('X shape:', X.shape)
print('y shape:', y.shape)

## 6. Add a channel dimension

TensorFlow Conv1D expects input shape:

`(samples, timesteps, channels)`

In [ ]:
X = X[..., np.newaxis]
print('New X shape:', X.shape)

## 7. Train/test split

We use a **time-based split** to avoid data leakage.

In [ ]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train mean:', y_train.mean())
print('y_test mean:', y_test.mean())

## 8. Build the 1D CNN

In [ ]:
model = models.Sequential([
    layers.Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(window_size, 1)),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(filters=64, kernel_size=3, activation='relu'),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

## 9. Compile the model

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## 10. Train

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

## 11. Plot training history

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.title('Accuracy')
plt.legend()

plt.show()

## 12. Evaluate on test data

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print('Test accuracy:', test_acc)

## 13. Generate predictions

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

## 14. Discussion questions

1. Why can a CNN work on a financial time series?
2. What does a 1D kernel detect in this context?
3. Why is a random train/test split inappropriate here?
4. Why might test accuracy be only slightly above 50%?
5. Does classification accuracy directly imply a profitable strategy?

## 15. Student experiments

Try the following modifications:

- change `kernel_size=3` to `kernel_size=5`
- increase `window_size` from 20 to 60
- add `Dropout(0.3)` before the final layer
- compare `GlobalAveragePooling1D()` with `Flatten()`
- replace SPY with `QQQ` or `^GSPC`


# Extension — Multi-feature 1D CNN

Now we add more realistic financial features:

- return
- rolling volatility
- moving average ratio
- volume change

In [ ]:
df2 = yf.download(ticker, start='2015-01-01', end='2025-01-01')

df2['return'] = df2['Close'].pct_change()
df2['volatility_5'] = df2['return'].rolling(5).std()
df2['ma_5'] = df2['Close'].rolling(5).mean()
df2['ma_20'] = df2['Close'].rolling(20).mean()
df2['ma_ratio'] = df2['ma_5'] / df2['ma_20']
df2['volume_change'] = df2['Volume'].pct_change()

df2 = df2.dropna()
df2.head()

In [ ]:
features = ['return', 'volatility_5', 'ma_ratio', 'volume_change']
data = df2[features].values
target_returns = df2['return'].values

window_size = 20

X_multi = []
y_multi = []

for i in range(window_size, len(df2) - 1):
    X_multi.append(data[i-window_size:i])
    y_multi.append(1 if target_returns[i+1] > 0 else 0)

X_multi = np.array(X_multi)
y_multi = np.array(y_multi)

print(X_multi.shape)
print(y_multi.shape)

In [ ]:
split = int(0.8 * len(X_multi))
X_train_m, X_test_m = X_multi[:split], X_multi[split:]
y_train_m, y_test_m = y_multi[:split], y_multi[split:]

model_multi = models.Sequential([
    layers.Conv1D(32, 3, activation='relu', input_shape=(window_size, X_multi.shape[2])),
    layers.MaxPooling1D(2),
    layers.Conv1D(64, 3, activation='relu'),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model_multi.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_multi.summary()

In [ ]:
history_multi = model_multi.fit(
    X_train_m,
    y_train_m,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
test_loss_m, test_acc_m = model_multi.evaluate(X_test_m, y_test_m)
print('Multi-feature test accuracy:', test_acc_m)

## Wrap-up

This exercise shows that **1D CNNs can be used on financial data**, but also highlights that:

- financial signals are weak and noisy
- avoiding leakage is essential
- small gains above chance may still be meaningful, depending on the strategy
- predictive accuracy is not the same as trading profitability
